<a href="https://colab.research.google.com/github/xyt556/I-GUIDE-GeoAI-Education/blob/main/notebooks/02-visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 交互式地图与可视化

## 简介

可视化在 GeoAI 工作流的每个阶段都至关重要，从训练前的检查图像和验证标签，到训练后的评估预测和沟通结果。

静态图表适用于简单的检查，但地理空间数据需要交互性，以便你可以平移、缩放、切换图层并并排比较数据集。

[leafmap](https://leafmap.org) 库提供了一个 Pythonic 接口，用于在 Jupyter 笔记本中构建交互式地图，并具有加载栅格、矢量和云端托管地理空间数据的便捷功能。

本教程涵盖了核心可视化技术：创建交互式地图、添加栅格和矢量图层、构建分屏对比图，以及在源图像上叠加模型预测。所有示例都使用了来自拉斯维加斯建筑物检测项目的数据集，包括 NAIP 航空影像、LiDAR 生成的地表高度 (HAG) 栅格、建筑物轮廓注释 (GeoJSON) 和栅格化的建筑物掩膜。

## 学习目标

通过本教程，你将能够：

- 使用 leafmap 创建交互式地图并自定义其外观
- 将栅格数据 (GeoTIFF, COG) 添加到地图中，并使用适当的配色方案和波段组合
- 使用自定义样式可视化矢量数据 (GeoJSON, GeoDataFrame)
- 构建分屏地图以比较数据集或不同时期的数据
- 在源图像上叠加模型预测以进行视觉评估
- 可视化来自 Microsoft Planetary Computer 的云端托管数据

## 安装

取消下面一行的注释以安装所需的软件包。

In [ ]:
# %pip install -U "geoai-py[extra]"

某些交互功能需要 leafmap。如果你在 I-GUIDE 中遇到 leafmap 错误，请取消下面几行的注释以安装 solara。

In [ ]:
# %pip install solara

然后，取消下面一行的注释以安装缺失的依赖项。

In [ ]:
# !/opt/conda/bin/python3.11 -m pip install ipyvuetify ipyvue bqplot ipyvolume ipywebrtc ipyleaflet

运行上述行后，刷新浏览器并重试。

In [ ]:
import os
import sys


def configure_proj_data(proj_dir: str) -> str:
    """Force PROJ's data directory through both env vars and library APIs.

    Setting ``PROJ_LIB``/``PROJ_DATA`` only takes effect if done before
    rasterio's C extension initializes PROJ. This helper additionally calls
    pyproj's ``set_data_dir`` as a belt-and-suspenders measure for builds
    that ignore the environment variables. It must still run before any CRS
    lookup is triggered (i.e. before ``leafmap`` / ``rio_tiler`` import).

    Args:
        proj_dir: Directory containing ``proj.db``.

    Returns:
        The configured PROJ data directory.

    Raises:
        AssertionError: If rasterio/leafmap were imported before this ran.
        FileNotFoundError: If ``proj.db`` is not present in ``proj_dir``.
    """
    assert "rasterio" not in sys.modules, "Restart the kernel; rasterio already loaded."
    if not os.path.exists(os.path.join(proj_dir, "proj.db")):
        raise FileNotFoundError(f"proj.db not in {proj_dir}")

    os.environ["PROJ_LIB"] = proj_dir
    os.environ["PROJ_DATA"] = proj_dir

    import pyproj

    pyproj.datadir.set_data_dir(proj_dir)
    return proj_dir


configure_proj_data(
    "/cvmfs/iguide.purdue.edu/software/conda/geoai-edu/lib/python3.12/site-packages/rasterio/proj_data"
)

In [ ]:
import os

os.environ["LOCALTILESERVER_CLIENT_PREFIX"] = "proxy/{port}"

## 使用 Leafmap 创建交互式地图

### 你的第一张地图

调用 `leafmap.Map()` 来创建一个具有平移、缩放和图层控制功能的交互式地图小部件。你可以配置地图中心、缩放级别和显示尺寸。

In [ ]:
import leafmap

m = leafmap.Map(center=[36.1617, -115.1524], zoom=11, height="600px")
m

`center` 参数接收一个 `[纬度, 经度]` 对，`zoom` 控制初始缩放级别，`height` 设置小部件的高度。

### 添加底图

底图提供了使地理空间可视化具有意义的地理上下文。`leafmap` 通过 `add_basemap()` 方法提供了对数百个底图的访问。

In [ ]:
m = leafmap.Map()
m.add_basemap("Esri.WorldImagery")
m

要查看所有可用的底图，请检查 `leafmap.basemaps` 字典：

In [ ]:
basemaps = list(leafmap.basemaps.keys())
print(f"Total basemaps: {len(basemaps)}")
print("First 10:", basemaps[:10])

你还可以添加多个底图并使用图层控件进行切换：

In [ ]:
m = leafmap.Map()
m.add_basemap("Esri.WorldImagery")
m.add_basemap("OpenTopoMap")
m

## 在地图上处理栅格数据

栅格数据构成了大多数 GeoAI 工作流的核心。`leafmap` 提供了几种将栅格数据添加到地图的方法，每种方法都针对不同的数据源和访问模式进行了优化。

### 添加 GeoTIFF 图层

`add_raster()` 方法加载本地或远程 GeoTIFF，并使用你选择的配色方案将其显示在地图上。在本教程中，我们使用了来自拉斯维加斯建筑物检测项目的四个数据集：

- **NAIP 航空影像** (4波段 GeoTIFF)：60 厘米分辨率的红、绿、蓝和近红外波段
- **地表高度 (HAG)** (单波段 GeoTIFF)：LiDAR 生成的以米为单位的地表高度
- **建筑物轮廓** (GeoJSON)：勾勒出单个建筑物轮廓的矢量多边形
- **建筑物掩膜** (单波段 GeoTIFF)：栅格化的二值掩膜，其中 1 表示建筑物像素，0 表示背景

In [ ]:
import geoai

我们定义每个数据集的 URL，并使用 `geoai.download_file()` 将其下载到本地：

In [ ]:
naip_url = "https://data.source.coop/opengeos/geoai/las-vegas-train-naip.tif"
hag_url = "https://data.source.coop/opengeos/geoai/las-vegas-train-hag.tif"
buildings_url = (
    "https://data.source.coop/opengeos/geoai/las-vegas-buildings-train.geojson"
)
buildings_mask_url = (
    "https://data.source.coop/opengeos/geoai/las-vegas-buildings-mask.tif"
)

In [ ]:
naip_path = geoai.download_file(naip_url)
hag_path = geoai.download_file(hag_url)
buildings_path = geoai.download_file(buildings_url)
mask_path = geoai.download_file(buildings_mask_url)

数据下载完成后，我们将 NAIP 图像添加为栅格图层：

In [ ]:
m = leafmap.Map()
m.add_raster(naip_path, layer_name="NAIP Image")
m

对于单波段栅格，你可以使用 `vmin` 和 `vmax` 指定配色方案和值范围。这里我们添加了地表高度栅格，将显示上限限制为 10 米，以便建筑物规模的结构能够脱颖而出：

In [ ]:
m.add_raster(
    hag_path, vmin=0, vmax=10, colormap="terrain", layer_name="Height Above Ground"
)

`add_raster()` 既接受本地文件路径也接受远程 URL。使用图层控件在图层之间切换。

### 云优化 GeoTIFF (COG)

云优化 GeoTIFF 允许你直接从远程服务器流式传输栅格数据，而无需下载整个文件。

In [ ]:
m = leafmap.Map()
m.add_cog_layer(naip_url, name="Las Vegas NAIP")
m

当处理托管在 Microsoft Planetary Computer 或 AWS Open Data 等云平台上的大型档案时，COG 特别有用。

### 波段组合

`add_raster()` 中的 `indexes` 参数允许你选择要显示的波段及其顺序。将近红外 (NIR) 波段放在红色通道中会使植被呈现亮红色，有助于区分植被区域和不透水表面：

In [ ]:
m = leafmap.Map()
m.add_raster(naip_path, indexes=[4, 1, 2], layer_name="False Color")
m

常见的 NAIP 波段组合包括：

- **真彩色** (波段 1, 2, 3)：自然外观
- **假彩色** (波段 4, 1, 2)：植被呈现亮红色，有助于区分建筑物和植被

## 可视化来自 Planetary Computer 的数据

Microsoft Planetary Computer 托管着数 PB 的地理空间数据，可通过 STAC API 访问。`geoai` 库提供了搜索、可视化和下载该目录中数据的便捷功能。

### 浏览可用集合

`pc_collection_list()` 函数以 DataFrame 的形式检索所有可用集合。

In [ ]:
collections = geoai.pc_collection_list()
print(f"Total collections: {len(collections)}")
collections.head(10)

### 搜索 STAC 条目

使用 `pc_stac_search()` 查找符合空间和时间标准的条目。这里我们搜索覆盖巴尔的摩的 NAIP 图像：

In [ ]:
naip_items = geoai.pc_stac_search(
    collection="naip",
    bbox=[-76.6657, 39.2648, -76.6478, 39.2724],
    time_range="2013-01-01/2014-12-31",
)
naip_items

你可以列出任何条目的可用资产：

In [ ]:
geoai.pc_item_asset_list(naip_items[0])

### 可视化 NAIP 图像

`view_pc_item()` 函数通过流式传输切片在交互式地图上渲染 STAC 条目，无需下载。

In [ ]:
geoai.view_pc_item(item=naip_items[0])

### 可视化土地覆盖数据

这里我们搜索切萨皮克湾高分辨率土地覆盖数据集，并使用分类配色方案显示它：

In [ ]:
lc_items = geoai.pc_stac_search(
    collection="chesapeake-lc-13",
    bbox=[-76.6657, 39.2648, -76.6478, 39.2724],
    time_range="2013-01-01/2014-12-31",
    max_items=10,
)
lc_items

In [ ]:
geoai.view_pc_item(item=lc_items[0], colormap_name="tab10", basemap="SATELLITE")

像土地覆盖这样的分类数据集非常适合使用定性配色方案，如 `"tab10"` 或 `"Set3"`。

### 可视化 Landsat 图像

对于像 Landsat 这样的多光谱数据，你可以选择特定波段或即时计算波段数学运算：

In [ ]:
landsat_items = geoai.pc_stac_search(
    collection="landsat-c2-l2",
    bbox=[-76.6657, 39.2648, -76.6478, 39.2724],
    time_range="2024-10-27/2024-12-31",
    query={"eo:cloud_cover": {"lt": 1}},
    max_items=10,
)
landsat_items

In [ ]:
geoai.pc_item_asset_list(landsat_items[0])

使用红、绿、蓝波段的真彩色合成：

In [ ]:
geoai.view_pc_item(item=landsat_items[0], assets=["red", "green", "blue"])

假彩色合成将近红外波段放在红色通道中，使植被呈现亮红色：

In [ ]:
geoai.view_pc_item(item=landsat_items[0], assets=["nir08", "red", "green"])

你还可以使用 `expression` 参数计算光谱指数：

In [ ]:
geoai.view_pc_item(
    item=landsat_items[0],
    expression="(nir08-red)/(nir08+red)",
    rescale="-1,1",
    colormap_name="greens",
    name="NDVI",
)

`expression` 参数接受使用资产名称作为变量的波段数学运算，`rescale` 控制值范围。计算在服务器端处理，因此不需要本地处理。

### 从 Planetary Computer 下载数据

使用 `pc_stac_download()` 将特定资产下载到磁盘：

In [ ]:
geoai.pc_stac_download(naip_items, output_dir="data", assets=["image", "thumbnail"])

你也可以直接将资产读取到 xarray DataArray 中而无需保存：

In [ ]:
ds = geoai.read_pc_item_asset(lc_items[0], asset="data")
ds

## 在地图上处理矢量数据

矢量数据在 GeoAI 中用作训练标签、参考边界和模型输出。`leafmap` 提供了灵活的方法，用于从本地文件、远程 URL 和内存中的 GeoDataFrames 添加矢量图层。

### 添加 GeoJSON 和 GeoDataFrame

你可以在工作流的任何阶段可视化来自 GeoJSON 文件、URL 或 GeoDataFrames 的矢量数据。

In [ ]:
import geopandas as gpd

gdf = gpd.read_file(buildings_path)
print(f"Features: {len(gdf)}")
gdf.head()

In [ ]:
m = leafmap.Map()
m.add_raster(naip_path)
m.add_gdf(gdf, layer_name="Building Footprints", zoom_to_layer=True)
m

你也可以直接从 URL 添加 GeoJSON，而无需先将其加载到 GeoDataFrame 中：

In [ ]:
m = leafmap.Map()
m.add_raster(naip_path)
m.add_geojson(buildings_path, layer_name="Buildings", zoom_to_layer=True)
m

### 设置矢量图层样式

自定义样式有助于区分要素类型并突出重要模式。你可以控制填充颜色、轮廓颜色、线宽和不透明度：

In [ ]:
style = {
    "color": "red",
    "weight": 2,
    "fillColor": "yellow",
    "fillOpacity": 0.3,
}
m = leafmap.Map()
m.add_raster(naip_path)
m.add_gdf(gdf, layer_name="Styled Buildings", style=style, zoom_to_layer=True)
m

`style` 字典遵循 Leaflet 路径选项约定：`color`（轮廓）、`weight`（轮廓宽度）、`fillColor` 和 `fillOpacity`。

### 添加标记和点

对于点数据，你可以向地图添加标记。对于密集的点图层，考虑使用标记聚合 (marker clustering) 以保持可读性。

In [ ]:
m = leafmap.Map(center=[36.1617, -115.1524], zoom=11)
m.add_marker(location=[36.1617, -115.1524])
m

## 用于对比的分屏地图

分屏地图提供了一种结构化的方式来比较数据集（例如不同日期的图像或预测结果与地面真值），而不会丢失空间上下文。

### 并排比较

`split_map()` 方法创建一个在两个图层之间带有可拖动分隔线的地图。这里我们将假彩色合成的 NAIP 图像与地表高度栅格进行比较：

In [ ]:
m = leafmap.Map()
m.split_map(
    left_layer=naip_path,
    right_layer=hag_path,
    left_args={"indexes": [4, 1, 2]},
    right_args={"vmin": 0, "vmax": 10, "cmap": "terrain"},
    left_label="NAIP",
    right_label="HAG",
)
m

左右拖动滑块以显示每个图层，并观察图像中的高大结构如何与高 HAG 值相对应。

## 可视化模型结果

对模型输出的视觉检查可以揭示模型在何处以及为何失败，从而揭示汇总统计数据无法捕捉到的空间误差模式。

### 叠加预测结果

将预测栅格或矢量输出叠加在源图像上，并设置部分透明度，以评估突出显示的要素是否对应于真实对象：

In [ ]:
m = leafmap.Map()
m.add_raster(naip_path, layer_name="NAIP Imagery")
m.add_raster(mask_path, opacity=0.8, nodata=0, layer_name="Building Mask")
m

将 `opacity` 设置为 1 以下可以让图像透出来，而 `nodata=0` 则让非建筑物像素变得透明。

### 比较标签与源图像

分屏地图也适用于在比较预测之前验证参考标签是否与源图像对齐：

In [ ]:
import leafmap

m = leafmap.Map()
m.split_map(
    left_layer=buildings_path,
    right_layer=naip_path,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
)
m

`geoai` 库还提供了 `create_split_map()`，可以在两个面板下方添加底图：

In [ ]:
geoai.create_split_map(
    left_layer=buildings_path,
    right_layer=naip_path,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
    basemap=naip_path,
)

拖动滑块以检查标注是否准确勾勒了建筑物边界、是否有建筑物被遗漏，或者是否有非建筑物结构被错误标记。

## GeoAI 可视化的最佳实践

**为你的数据类型选择合适的配色方案。** 顺序配色方案（如 `"viridis"` 或 `"terrain"`）适用于高程或 NDVI 等连续数据。发散配色方案（如 `"RdBu"`）更适合具有意义中心点的数据，如温度异常或变化值。分类配色方案应为每个类别使用截然不同、易于区分的颜色。避免使用彩虹配色方案，因为它们缺乏自然的感知顺序，可能会误导读者。

**始终包含空间上下文。** 没有底图或参考特征的预测图很难解释。应包含卫星图像、行政边界或其他参考图层，以帮助查看者定位并了解结果的地理背景。

**在对比中使用一致的样式。** 比较来自不同实验或不同时期的模型输出时，所有图层应使用相同的配色方案、值范围和不透明度设置。不一致的样式可能会产生视觉上的虚假差异，而这些差异并不能反映实际的数据差异。

**清晰地标记图层。** 为每个图层提供一个显示在图层控件中的描述性名称。当你同时激活多个图层时，像“建筑物预测 (U-Net)”这样的名称比“图层 1”有用得多。

**考虑你的受众。** 技术同事可能会喜欢具有完整交互功能的详细多图层地图，而决策者可能需要带有清晰图例和注释的更简单的可视化。根据将使用你的可视化的受众来定制其复杂性和格式。

## 关键要点

1. 交互式地图揭示了汇总统计数据无法捕捉的空间模式，使可视化在整个 GeoAI 生命周期中至关重要。
2. `leafmap` 提供了一个高级 Python 接口，用于在 Jupyter 笔记本中创建交互式地理空间地图，并内置了对底图、栅格图层和矢量叠加的支持。
3. 栅格可视化支持通过 `add_raster()` 访问本地 GeoTIFF 和通过 `add_cog_layer()` 访问远程 COG，并具有可自定义的配色方案、波段组合和值范围。
4. 矢量可视化处理 GeoJSON 文件、GeoDataFrames 和点标记，具有灵活的样式，可用于在图像上叠加注释。
5. 分屏地图通过可拖动滑块实现并排比较，用于评估预测、比较图像和验证注释。
6. 模型结果叠加将预测栅格与源图像相结合，使用 `nodata` 实现透明度，使用 `opacity` 实现图层混合。
7. 通过 `geoai` 实现的 Planetary Computer 集成让你能够搜索、预览和下载云端托管的数据集，并实现波段组合和光谱指数的服务器端渲染。
8. 一致的样式和清晰的标注可以提高可视化的可读性和有效性。